# Uyum eğitimi: sağ gluteus medius + maksimus %20

Bu defter, yayımlanmış yürüyüş kontrolcüsünü sağ gluteus medius ve gluteus maksimus kasları
%20 güçteyken yeniden eğitir. Ardından aynı testlerle değerlendirir.

**Soru:** Kontrolcü zayıflığa alışırsa model yürüyebiliyor mu?
Alışmamış kontrolcüyle sonuç 6 denemede 0 yürüme, 1 sapma, 5 düşmeydi.

**Nasıl çalıştırılır**
1. Üst menüden **Çalışma zamanı > Tümünü çalıştır**.
2. Google Drive izni istenince onay verin. Sonuçlar ve ara kayıtlar Drive'da
   `kas_animasyon_colab` klasörüne yazılır. Oturum kapanırsa hiçbir şey kaybolmaz.
3. Oturum kapanırsa defteri yeniden açıp tekrar **Tümünü çalıştır** deyin.
   Eğitim son kayıt noktasından devam eder.

**Süre (tahmin):** kurulum 5–10 dk, eğitim 2–3 saat, değerlendirme 20–30 dk.
Eğitim sırasında bu sekme açık kalmalı.

In [ ]:
# 1) Kurulum: Drive, kaynaklar, paketler
import hashlib, io, json, os, subprocess, sys, urllib.request, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/kas_animasyon_colab')
(DRIVE / 'adaptation').mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/kas_animasyon')
(PROJECT / 'scripts').mkdir(parents=True, exist_ok=True)
(PROJECT / 'results').mkdir(exist_ok=True)
link = PROJECT / 'results' / 'adaptation'
if not link.exists():
    link.symlink_to(DRIVE / 'adaptation')

SCRIPTS = {"simulate_weakness_sweep.py": "\"\"\"Forward-simulation sweep: weaken muscle groups and let the published controller walk.\n\nEvery scenario starts from the same deterministic state as the baseline and\nuses the same controller without retraining. Only the active force component\n(gainprm[2]) of the chosen muscles is scaled; passive forces stay unchanged.\n\nThe environment's own termination (mean coordinate error above qpos_diff_th)\nis switched off so every run lasts three gait cycles unless the model falls.\nOutcomes are classified afterwards:\n  - walked:   mean coordinate error stayed under the controller threshold\n  - deviated: threshold crossed but the pelvis stayed up\n  - fell:     pelvis height dropped below FALL_RATIO of its initial value\n\"\"\"\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nfrom itertools import combinations\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time as clock\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nGROUPS = {\n    'right_tibialis_anterior': ['tibant_r'],\n    'right_soleus': ['soleus_r'],\n    'right_gastrocnemius': ['gasmed_r', 'gaslat_r'],\n    'right_vasti': ['vasint_r', 'vaslat_r', 'vasmed_r'],\n    'right_hamstrings': ['bflh_r', 'semimem_r', 'semiten_r'],\n    'right_gluteus_medius': ['glmed1_r', 'glmed2_r', 'glmed3_r'],\n    'right_gluteus_maximus': ['glmax1_r', 'glmax2_r', 'glmax3_r'],\n    'right_hip_flexors': ['iliacus_r', 'Ps_L1_VB_r', 'Ps_L1_TP_r', 'Ps_L1_L2_IVD_r', 'Ps_L2_TP_r',\n                          'Ps_L2_L3_IVD_r', 'Ps_L3_TP_r', 'Ps_L3_L4_IVD_r', 'Ps_L4_TP_r',\n                          'Ps_L4_L5_IVD_r', 'Ps_L5_TP_r', 'Ps_L5_VB_r'],\n}\nTHRESHOLD = 0.06   # controller's training threshold, rad (mean |qpos - ref|)\nFALL_RATIO = 0.7\n_worker = {}\n\n\ndef init_worker(model_path=None, env_path=None):\n    os.environ.setdefault('OMP_NUM_THREADS', '1')\n    import torch\n    torch.set_num_threads(1)\n    sys.path.insert(0, str(ROOT / 'vendor/msgym/DynSyn-SAC/SB3-Scripts'))\n    import msgym  # noqa: F401  registers the environments\n    from DynSyn import SAC_DynSyn\n    checkpoint = ROOT / 'vendor/checkpoints/LocomotionLegs'\n    _worker['config'] = json.loads((checkpoint / 'locomotionLegs.json').read_text())\n    # Inference only: skip the ~2.8 GiB training replay buffer.\n    _worker['policy'] = SAC_DynSyn.load(str(model_path or checkpoint / 'checkpoint/best_model.zip'), device='cpu',\n                                        custom_objects={'buffer_size': 1})\n    _worker['env_path'] = str(env_path or checkpoint / 'checkpoint/best_env.zip')\n\n\ndef rollout(job):\n    import gymnasium as gym\n    import mujoco\n    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize\n    from wrapper import MuscleNormWrapper\n\n    groups, capacity, seed, noise = job\n    config, policy = _worker['config'], _worker['policy']\n    options = dict(config['single_env_kwargs'])\n    options.update(random_init=False, reset_noise_scale=noise, gait_cycles=3, qpos_diff_th=1e9)\n    env = MuscleNormWrapper(gym.make(config['env_name'], render_mode=None, **options))\n    base = env.unwrapped\n    model, data = base.model, base.data\n    normalizer = VecNormalize.load(_worker['env_path'], DummyVecEnv([lambda: env]))\n    normalizer.training = False\n    normalizer.norm_reward = False\n    muscles = [m for g in groups for m in GROUPS[g]]\n    ids = [model.actuator(m).id for m in muscles]\n    passive = model.actuator_biasprm.copy()\n    if np.any(model.actuator_gainprm[ids, 2] <= 0):\n        raise ValueError('Expected explicit active force parameter')\n    model.actuator_gainprm[ids, 2] *= capacity\n    if not np.array_equal(passive, model.actuator_biasprm):\n        raise ValueError('Passive parameters changed')\n    if noise > 0:\n        normalizer.seed(seed)\n    observation = normalizer.reset()\n    pelvis = model.body('pelvis').id\n    start_height = float(data.xpos[pelvis, 2])\n    times, errors, heights = [], [], []\n    end_time = 3 * base.terminate_time\n    try:\n        while True:\n            action, _ = policy.predict(observation, deterministic=True)\n            observation, _, done, _ = normalizer.step(action)\n            if done[0]:\n                break\n            times.append(float(data.time))\n            errors.append(float(np.abs(data.qpos[3:] - base.qpos_ref[3:]).mean()))\n            heights.append(float(data.xpos[pelvis, 2]))\n            if heights[-1] < FALL_RATIO * start_height:\n                break\n    finally:\n        normalizer.close()\n    times, errors, heights = map(np.asarray, (times, errors, heights))\n    over = np.flatnonzero(errors > THRESHOLD)\n    fell = heights.size and heights.min() < FALL_RATIO * start_height\n    outcome = 'fell' if fell else ('deviated' if over.size else 'walked')\n    return {\n        'groups': list(groups), 'remaining_active_capacity': capacity, 'outcome': outcome,\n        'seed': seed, 'reset_noise_scale': noise,\n        'simulated_time_s': float(times[-1]) if times.size else 0.0,\n        'requested_time_s': float(end_time),\n        'first_threshold_crossing_s': float(times[over[0]]) if over.size else None,\n        'max_mean_coordinate_error_rad': float(errors.max()),\n        'mean_coordinate_error_rad': float(errors.mean()),\n        'min_pelvis_height_ratio': float(heights.min() / start_height),\n        'error_trace': errors[::5].round(5).tolist(),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--levels', type=float, nargs='+', default=[0.6, 0.4, 0.2, 0.0])\n    parser.add_argument('--workers', type=int, default=max(1, (os.cpu_count() or 2) - 2))\n    parser.add_argument('--singles-only', action='store_true')\n    parser.add_argument('--output', default='results/forward_sweep')\n    parser.add_argument('--scenario', nargs='+', help='Groups joined by +; replaces the default grid')\n    parser.add_argument('--seeds', type=int, default=1, help='Repeats with reset noise (seed 0..n-1)')\n    parser.add_argument('--noise', type=float, default=0.0, help='Reset noise scale (training used 0.001)')\n    args = parser.parse_args()\n    scenarios = [(g,) for g in GROUPS]\n    if not args.singles_only:\n        scenarios += list(combinations(GROUPS, 2))\n    if args.scenario:\n        scenarios = [tuple(v.split('+')) for v in args.scenario]\n        if any(g not in GROUPS for s in scenarios for g in s):\n            raise ValueError('Unknown group')\n    seeds = range(args.seeds)\n    jobs = [((), 1.0, k, args.noise) for k in seeds] +            [(s, c, k, args.noise) for c in args.levels for s in scenarios for k in seeds]\n    output = ROOT / args.output\n    output.mkdir(parents=True, exist_ok=True)\n    results, started = [], clock.time()\n    with ProcessPoolExecutor(args.workers, initializer=init_worker) as pool:\n        for i, result in enumerate(pool.map(rollout, jobs), 1):\n            results.append(result)\n            print(f\"[{i}/{len(jobs)} {clock.time() - started:.0f}s] {result['remaining_active_capacity']:.2f} s{result['seed']} \"\n                  f\"{'+'.join(result['groups']) or 'baseline'}: {result['outcome']} \"\n                  f\"(max err {result['max_mean_coordinate_error_rad']:.3f}, t={result['simulated_time_s']:.2f})\",\n                  flush=True)\n            report = {'scope': 'Published LocomotionLegs controller, deterministic start, active force only, no retraining',\n                      'groups': GROUPS, 'threshold_rad': THRESHOLD, 'fall_ratio': FALL_RATIO,\n                      'interpretation': ('Outcomes describe this controller without adaptation. A failure does not '\n                                         'prove that a person or another controller could not walk.'),\n                      'results': results}\n            (output / 'sweep.json').write_text(json.dumps(report, indent=2), encoding='utf-8')\n\n\nif __name__ == '__main__':\n    main()\n", "adapt_controller.py": "\"\"\"Fine-tune the published controller on a weakened model (adaptation experiment).\n\nQuestion: if the controller is allowed to adapt to the weakness, can it keep\nwalking? Starting point is the published LocomotionLegs checkpoint; the\nenvironment, reward and termination are the original training settings, so\nthe controller is still rewarded for tracking the normal reference gait.\nOnly the active force of the chosen muscle groups is scaled.\n\n  python scripts/adapt_controller.py --groups right_gluteus_medius right_gluteus_maximus \\\n      --capacity 0.2 --steps 200000 --envs 4 --name glut_020\n\"\"\"\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nSB3 = ROOT / 'vendor/msgym/DynSyn-SAC/SB3-Scripts'\nsys.path.insert(0, str(SB3))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nCHECKPOINT = ROOT / 'vendor/checkpoints/LocomotionLegs'\n\n\ndef make_env(config, groups, capacity, seed):\n    def factory():\n        import gymnasium as gym\n        import msgym  # noqa: F401\n        from stable_baselines3.common.monitor import Monitor\n        from wrapper import MuscleNormWrapper\n        import simulate_weakness_sweep as sweep\n        env = gym.make(config['env_name'], render_mode=None, **config['single_env_kwargs'])\n        model = env.unwrapped.model\n        ids = [model.actuator(m).id for g in groups for m in sweep.GROUPS[g]]\n        model.actuator_gainprm[ids, 2] *= capacity\n        env = Monitor(MuscleNormWrapper(env))\n        env.reset(seed=seed)\n        return env\n    return factory\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--groups', nargs='+', required=True)\n    parser.add_argument('--capacity', type=float, required=True)\n    parser.add_argument('--steps', type=int, default=200_000)\n    parser.add_argument('--envs', type=int, default=4)\n    parser.add_argument('--buffer', type=int, default=50_000)\n    parser.add_argument('--learning-starts', type=int, default=4_000)\n    parser.add_argument('--lr', type=float, default=1e-4)\n    parser.add_argument('--save-every', type=int, default=25_000)\n    parser.add_argument('--name', required=True)\n    parser.add_argument('--seed', type=int, default=0)\n    parser.add_argument('--resume', action='store_true',\n                        help='continue from the latest model_<n>.zip in the output folder')\n    args = parser.parse_args()\n\n    import torch\n    torch.set_num_threads(2)\n    import msgym  # noqa: F401\n    from stable_baselines3.common.callbacks import BaseCallback\n    from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize\n    from DynSyn import SAC_DynSyn\n    import simulate_weakness_sweep as sweep\n\n    if any(g not in sweep.GROUPS for g in args.groups):\n        raise ValueError('Unknown group')\n    config = json.loads((CHECKPOINT / 'locomotionLegs.json').read_text())\n    out = ROOT / 'results/adaptation' / args.name\n    out.mkdir(parents=True, exist_ok=True)\n    (out / 'config.json').write_text(json.dumps(vars(args), indent=2), encoding='utf-8')\n\n    # Start from the published checkpoint, or from the latest saved one with --resume.\n    # The replay buffer is not saved, so a resumed run refills it first.\n    offset = 0\n    model_file = CHECKPOINT / 'checkpoint/best_model.zip'\n    env_file = CHECKPOINT / 'checkpoint/best_env.zip'\n    saved = sorted(int(p.stem.split('_')[1]) for p in out.glob('model_*.zip'))\n    if args.resume and saved:\n        offset = saved[-1]\n        model_file, env_file = out / f'model_{offset}.zip', out / f'env_{offset}.zip'\n        print(f'Resuming from step {offset}', flush=True)\n    if offset >= args.steps or (args.resume and (out / 'final_model.zip').exists()):\n        print('Already finished', flush=True)\n        return\n\n    vec = SubprocVecEnv([make_env(config, args.groups, args.capacity, args.seed + offset + i)\n                         for i in range(args.envs)])\n    vec = VecNormalize.load(str(env_file), vec)\n    vec.training = True          # observation statistics may follow the new gait\n    vec.norm_reward = False\n    model = SAC_DynSyn.load(str(model_file), env=vec, device='cpu',\n                            custom_objects={'buffer_size': args.buffer, 'learning_starts': args.learning_starts,\n                                            'learning_rate': args.lr, 'lr_schedule': lambda _: args.lr,\n                                            'tensorboard_log': None})\n    model.learning_rate = args.lr\n    model._setup_lr_schedule()\n\n    class Log(BaseCallback):\n        def __init__(self):\n            super().__init__()\n            self.start = time.time()\n            progress = out / 'progress.json'\n            self.rows = [r for r in json.loads(progress.read_text()) if r['steps'] <= offset] \\\n                if offset and progress.exists() else []\n            self.next_save = offset + args.save_every\n\n        def _on_step(self):\n            n = offset + self.num_timesteps\n            if n >= self.next_save:\n                self.next_save += args.save_every\n                self.model.save(out / f'model_{n}.zip')\n                vec.save(str(out / f'env_{n}.zip'))\n            if self.n_calls % 250 == 0:\n                eps = list(self.model.ep_info_buffer)[-20:]\n                row = {'steps': int(n), 'seconds': round(time.time() - self.start, 1),\n                       'steps_per_s': round(self.num_timesteps / max(1e-9, time.time() - self.start), 1),\n                       'episode_length_mean': float(np.mean([e['l'] for e in eps])) if eps else None,\n                       'episode_reward_mean': float(np.mean([e['r'] for e in eps])) if eps else None}\n                self.rows.append(row)\n                (out / 'progress.json').write_text(json.dumps(self.rows, indent=1), encoding='utf-8')\n                print(json.dumps(row), flush=True)\n            return True\n\n    model.num_timesteps = 0\n    model.learn(total_timesteps=args.steps - offset, callback=Log(), reset_num_timesteps=True, log_interval=10_000)\n    model.save(out / 'final_model.zip')\n    vec.save(str(out / 'final_env.zip'))\n    vec.close()\n\n\nif __name__ == '__main__':\n    os.environ.setdefault('OMP_NUM_THREADS', '1')\n    main()\n", "evaluate_adaptation.py": "\"\"\"Evaluate an adapted controller with the same rollouts as the forward sweep.\n\nFor a checkpoint from adapt_controller.py, run the weakness it was trained on,\neach single group alone, and the healthy model. Each scenario: one\ndeterministic start plus seeds with the training reset noise (0.001).\nOutcomes use the sweep's classes (walked / deviated / fell).\n\n  python scripts/evaluate_adaptation.py --name glut_med_max_020 --step 100000\n  python scripts/evaluate_adaptation.py --name glut_med_max_020 --step final\n\"\"\"\nimport argparse\nfrom concurrent.futures import ProcessPoolExecutor\nimport json\nfrom pathlib import Path\nimport sys\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport simulate_weakness_sweep as sweep  # noqa: E402\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--name', required=True)\n    parser.add_argument('--step', default='final', help=\"checkpoint step, 'final', or 'original'\")\n    parser.add_argument('--seeds', type=int, default=5)\n    parser.add_argument('--workers', type=int, default=4)\n    args = parser.parse_args()\n    folder = ROOT / 'results/adaptation' / args.name\n    config = json.loads((folder / 'config.json').read_text())\n    groups, capacity = tuple(config['groups']), config['capacity']\n    if args.step == 'original':\n        model_path = env_path = None\n    elif args.step == 'final':\n        model_path, env_path = folder / 'final_model.zip', folder / 'final_env.zip'\n    else:\n        model_path, env_path = folder / f'model_{args.step}.zip', folder / f'env_{args.step}.zip'\n    scenarios = [(groups, capacity)] + [((g,), capacity) for g in groups] + [((), 1.0)]\n    jobs = [(s, c, 0, 0.0) for s, c in scenarios] + \\\n           [(s, c, k, 0.001) for s, c in scenarios for k in range(args.seeds)]\n    with ProcessPoolExecutor(args.workers, initializer=sweep.init_worker,\n                             initargs=(model_path and str(model_path), env_path and str(env_path))) as pool:\n        results = list(pool.map(sweep.rollout, jobs))\n    summary = {}\n    for r in results:\n        key = ('+'.join(g.replace('right_', '') for g in r['groups']) or 'normal') + f\" @{r['remaining_active_capacity']:.2f}\"\n        entry = summary.setdefault(key, {'walked': 0, 'deviated': 0, 'fell': 0, 'max_errors': []})\n        entry[r['outcome']] += 1\n        entry['max_errors'].append(round(r['max_mean_coordinate_error_rad'], 3))\n    report = {'checkpoint': args.step, 'trained_on': {'groups': list(groups), 'capacity': capacity},\n              'summary': summary,\n              'results': [{k: v for k, v in r.items() if k != 'error_trace'} for r in results]}\n    out = folder / f'eval_{args.step}.json'\n    out.write_text(json.dumps(report, indent=2), encoding='utf-8')\n    for key, e in summary.items():\n        print(f\"{key:45s} walked {e['walked']}  deviated {e['deviated']}  fell {e['fell']}   max err {e['max_errors']}\")\n\n\nif __name__ == '__main__':\n    main()\n"}
for name, text in SCRIPTS.items():
    (PROJECT / 'scripts' / name).write_text(text, encoding='utf-8')


def unpack(url, sha256, destination, strip_top=True):
    if destination.exists() and any(destination.iterdir()):
        return
    with urllib.request.urlopen(url, timeout=300) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != sha256:
        raise ValueError(f'Checksum mismatch: {url}')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts[1 if strip_top else 0:]
            if not parts or member.is_dir():
                continue
            if any(p in ('.', '..') for p in parts):
                raise ValueError('Unsafe archive member')
            target = destination.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))


SOURCES = [{"repository": "LNSGroup/msgym", "revision": "ad3aac166dab3577d7d16b4a03faccef94c7aac7", "archive_url": "https://codeload.github.com/LNSGroup/msgym/zip/ad3aac166dab3577d7d16b4a03faccef94c7aac7", "archive_sha256": "37dad71e1e79c3c53169ae4520134a9aedde8710afc4f4bf15a3491e8ee56c76", "archive_bytes": 20540062, "directory": "vendor\\msgym"}, {"repository": "LNSGroup/MS-Human-700", "revision": "2d686957aefd5739cf4d2859a6acfd94c8f84400", "archive_url": "https://codeload.github.com/LNSGroup/MS-Human-700/zip/2d686957aefd5739cf4d2859a6acfd94c8f84400", "archive_sha256": "65975d132606149e7707c8ce0b6015c33e2d6fd50d15f00e416f961eb47327bc", "archive_bytes": 30783005, "directory": "vendor\\MS-Human-700"}]
vendor = PROJECT / 'vendor'
unpack(SOURCES[0]['archive_url'], SOURCES[0]['archive_sha256'], vendor / 'msgym')
unpack(SOURCES[1]['archive_url'], SOURCES[1]['archive_sha256'], vendor / 'msgym' / 'msgym' / 'MS-Human-700')
unpack('https://github.com/LNSGroup/msgym/releases/download/Checkpoints/LocomotionLegs.zip',
       '8f442634f9e90cf8cd38de22f74fc0da0b9d842b726cbaf812161533e82313f4',
       vendor / 'checkpoints', strip_top=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'mujoco==3.17.0', 'gymnasium==1.2.3',
                'stable-baselines3==2.7.1'], check=True)
paths = [str(vendor / 'msgym'), str(vendor / 'msgym' / 'DynSyn-SAC'),
         str(vendor / 'msgym' / 'DynSyn-SAC' / 'SB3-Scripts')]
os.environ['PYTHONPATH'] = os.pathsep.join(paths + [os.environ.get('PYTHONPATH', '')])
sys.path[:0] = paths
os.chdir(PROJECT)
print('Python', sys.version.split()[0], '| işlemci çekirdeği:', os.cpu_count())
print('Kurulum tamam.')

In [ ]:
# 2) Hızlı kontrol: alışmamış kontrolcü, sağlıklı model, tek yürüyüş
import time
sys.path.insert(0, str(PROJECT / 'scripts'))
import simulate_weakness_sweep as sweep
sweep.init_worker()
start = time.time()
check = sweep.rollout(((), 1.0, 0, 0.0))
print('Sonuç:', check['outcome'], '| en büyük hata:', round(check['max_mean_coordinate_error_rad'], 3),
      'rad | süre:', round(time.time() - start), 's')
if check['outcome'] != 'walked':
    raise RuntimeError('Sağlıklı model yürümedi, kurulumda sorun var. Bu çıktıyı paylaşın.')

In [ ]:
# 3) Uyum eğitimi (60 bin adım, her 10 binde bir Drive'a kayıt)
envs = max(1, min(4, os.cpu_count()))
command = [sys.executable, 'scripts/adapt_controller.py',
           '--groups', 'right_gluteus_medius', 'right_gluteus_maximus', '--capacity', '0.2',
           '--steps', '60000', '--envs', str(envs), '--buffer', '50000', '--learning-starts', '300',
           '--save-every', '10000', '--name', 'glut_med_max_020', '--resume']
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
if process.wait() != 0:
    raise RuntimeError('Eğitim hata verdi. Yukarıdaki çıktıyı paylaşın.')
print('Eğitim tamam.')

In [ ]:
# 4) Değerlendirme: eğitilmiş kontrolcü, aynı testler
BASELINE = {"gluteus_medius+gluteus_maximus @0.20": {"walked": 0, "deviated": 1, "fell": 5}, "gluteus_medius @0.20": {"walked": 6, "deviated": 0, "fell": 0}, "gluteus_maximus @0.20": {"walked": 6, "deviated": 0, "fell": 0}, "normal @1.00": {"walked": 6, "deviated": 0, "fell": 0}}
workers = max(1, os.cpu_count())
subprocess.run([sys.executable, 'scripts/evaluate_adaptation.py', '--name', 'glut_med_max_020',
                '--step', 'final', '--seeds', '5', '--workers', str(workers)], check=True)
report = json.loads((DRIVE / 'adaptation/glut_med_max_020/eval_final.json').read_text())
print('\nSenaryo                                   önce (y/s/d)    sonra (y/s/d)')
for key, after in report['summary'].items():
    before = BASELINE.get(key, {})
    b = '/'.join(str(before.get(o, '-')) for o in ('walked', 'deviated', 'fell'))
    a = '/'.join(str(after[o]) for o in ('walked', 'deviated', 'fell'))
    print(f'{key:42s} {b:15s} {a}')
print('\ny = yürüdü, s = saptı, d = düştü')

In [ ]:
# 5) Sonuçları zip olarak indir (Drive'da da duruyor)
import shutil
from google.colab import files
archive = shutil.make_archive('/content/uyum_sonuclari', 'zip', root_dir=DRIVE / 'adaptation')
files.download(archive)